# Bone-Loss Segmentation - Inference Notebook

**Affiliation:** [InReDD](https://inredd.com.br/en)

This notebook runs the full inference pipeline trained in the companion *Training*
notebook against intraoral panoramic radiographs. Stages, in order of execution:

1. **Mouth detection** (Detectron2 RetinaNet) - locate and crop the mouth region.
2. **Tooth instance segmentation** (Detectron2 Mask R-CNN) - extract per-tooth polygons.
3. **Crown and alveolar-ridge segmentation** (YOLOv11x-seg) - locate crowns and the alveolar ridge.
4. **Bone-loss measurement** - geometric analysis comparing tooth polygons against crown and ridge segmentations.
5. **LabelMe export and visualization** - produce annotated images and a LabelMe-compatible JSON.

A Gradio interface is provided for interactive single-image analysis; a debug
section at the end of the notebook exposes a per-image batch runner for
methodological inspection.

## Data and Model Availability

> The pretrained model weights and trained checkpoints referenced in this notebook are property of **InReDD** (https://inredd.com.br/en) and are **not publicly available**, nor distributed with this notebook. The clinical dataset is also property of InReDD; it may be requested for non-commercial academic use through the [InReDD Open Data portal](https://inredd.com.br/en/solutions/open-data). This notebook is released for **methodological transparency and reproducibility only**.

## Reproducibility

- Tested on a local workstation with an NVIDIA RTX 5070 12 GB GPU.
- Key dependencies (pinned in the install cell): `pytorch-lightning==2.4.0`, `transformers==4.46.2`, `ultralytics`, plus Detectron2 built from source.

## Repository layout (expected for re-running locally)

```
boneloss_inference/
|-- data/
|-- models/
|   |-- inredd_mouth_detector.pth              # InReDD only (Detectron2 RetinaNet)
|   |-- inredd_tooth_segmenter.pth             # InReDD only (Detectron2 Mask R-CNN)
|   |-- inredd_crown_segmenter.pt              # InReDD only (YOLOv11)
|   |-- inredd_alveolar_ridge_segmenter.pt     # InReDD only (YOLOv11)
|-- notebooks/
    |-- Bone_Loss_Segmentation_(Inference).ipynb
```


# 1. Environment Setup

This cell installs the Python dependencies, downloads the InReDD model weights from Google Drive, and compiles Detectron2 from source. The runtime is killed at the end so the kernel restarts and picks up the freshly compiled extensions.

> **Run this cell only once.** After the kernel restart, skip directly to section 2.


In [ ]:
# NOTE: InReDD property - not publicly available, not distributed with this notebook.
print("Installing Python libraries...")
!pip install -q gdown pytorch-lightning==2.4.0
!pip install -q transformers==4.46.2 datasets==2.21.0 ultralytics
print("Libraries installed.")

print("\nDownloading datasets and models from Google Drive...")

# --- Dataset Bone Loss ---
# Use -O for a fixed filename and -d to extract into a specific folder
DATASET_ZIP_PATH = '/content/inredd_crown_alveolar_ridge_dataset.zip'
DATASET_EXTRACT_PATH = '/content/Datasets/'
!gdown --id '<INREDD_DATASET_DRIVE_ID>'
!mkdir -p {DATASET_EXTRACT_PATH}
!unzip -q -o {DATASET_ZIP_PATH} -d {DATASET_EXTRACT_PATH}
print(f"Dataset 'Bone Loss' downloaded and extracted to: {DATASET_EXTRACT_PATH}")

# --- Mouth-Detection Models ---
!gdown --id '<INREDD_MOUTH_DETECTOR_DRIVE_ID>' --quiet
!gdown --id '<INREDD_TOOTH_SEGMENTER_DRIVE_ID>' --quiet
print("Mouth-detection models downloaded.")

# --- Tooth-Segmentation Models ---
!gdown --id '<INREDD_CROWN_SEGMENTER_DRIVE_ID>' --quiet
!gdown --id '<INREDD_ALVEOLAR_RIDGE_SEGMENTER_DRIVE_ID>' --quiet
print("Tooth-segmentation models downloaded.")

# --- Bone-Loss Models ---
!gdown --id '<INREDD_BONELOSS_CROWN_DRIVE_ID>' --quiet
!gdown --id '<INREDD_BONELOSS_RIDGE_DRIVE_ID>' --quiet
print("Bone-loss models downloaded.")

print("\nOrganizing model folder structure...")
!mkdir -p Models/MouthDetection
!mv inredd_mouth_detector.* Models/MouthDetection/
print("Models moved to folder 'Models/MouthDetection/'.")

!mkdir -p Models/TeethSegmentation
!mv inredd_tooth_segmenter.* Models/TeethSegmentation/
print("Models moved to folder 'Models/TeethSegmentation/'.")

!mkdir -p Models/BoneLossSegmentation
!mv inredd_crown_segmenter.* Models/BoneLossSegmentation/
!mv inredd_alveolar_ridge_segmenter.* Models/BoneLossSegmentation/
print("Models moved to folder 'Models/BoneLossSegmentation/'.")

print("\nCompiling and installing Detectron2 (may take a few minutes)...")

!git clone https://github.com/facebookresearch/detectron2.git > /dev/null 2>&1

# Install Detectron2 from the cloned folder, compiling C++/CUDA extensions
!pip install -e detectron2

import os

try:
    import detectron2
    print("\nDetectron2 imported successfully!")
    print("Setup complete! The session will now restart to apply the changes...")

    os.kill(os.getpid(), 9)
except ImportError:
    print("\nFailed to import Detectron2. Installation may have failed.")

# 2. Code and Model Preparation

After the environment is installed and the kernel has restarted, this section imports the libraries, defines the pipeline classes and helper functions, and loads the trained model weights into memory.


## 2.1. Library Imports

Imports for all dependencies used across the pipeline:

- **Deep learning / computer vision:** PyTorch, Detectron2, Ultralytics (YOLO), HuggingFace Transformers, PyTorch Lightning.
- **Numerical / geometric:** NumPy, Pandas, Shapely (polygon area and topology).
- **UI / visualization:** Gradio (interactive interface), Matplotlib, OpenCV.
- **Utilities:** `os`, `json`, `base64`, `pathlib`.


In [ ]:
import os
import cv2
import abc
import json
import types
import torch
import base64
import shutil
import random
import zipfile
import traceback
import numpy as np
import gradio as gr
import pandas as pd
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import torch.multiprocessing as mp

from torch import nn
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO
from typing import List, Dict
from dotenv import load_dotenv
from PIL import Image, ImageDraw
from datasets import load_metric
from detectron2 import model_zoo
from shapely.ops import unary_union
from matplotlib import pyplot as plt
from detectron2.config import get_cfg
from detectron2.structures import Boxes
from detectron2.structures import Instances
from detectron2.data import MetadataCatalog
from shapely.errors import TopologicalError
from detectron2.engine import DefaultPredictor
from detectron2.modeling import GeneralizedRCNN
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import Dataset, DataLoader
from shapely.geometry import Polygon, MultiPolygon
from detectron2.utils.visualizer import Visualizer
from detectron2.layers import paste_masks_in_image
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks.model_checkpoint import ModelCheckpoint
from transformers import SegformerFeatureExtractor, SegformerForSemanticSegmentation
from detectron2.export.torchscript import scripting_with_instances, freeze_training_mode

load_dotenv()
mp.set_start_method('spawn', force=True)
torch.set_num_threads(1)

## 2.2. Model Loading and Helper Functions

This cell defines the pipeline classes (`MouthDetector`, `InferenciaSegmentacaoDeDentes`, `BoneLossSegmentation`), the bone-loss measurement function (`analisar_perda_ossea`), the LabelMe exporter (`exportar_para_labelme`), and the OpenCV / Matplotlib visualizers, then instantiates the top-level segmentators with the InReDD weights.


In [ ]:
# NOTE: InReDD property - not publicly available, not distributed with this notebook.
class MouthDetector:
    """Detect a mouth bounding box in intraoral photographs using Detectron2 RetinaNet.

    Wraps a Detectron2 ``DefaultPredictor`` built from a base RetinaNet config
    and trained weights provided as a ``.pth`` file. Used as a preprocessing
    crop step before tooth segmentation.
    """
    def __init__(self, weight_path: str):
        """Build the underlying ``DefaultPredictor`` from the trained RetinaNet weights.

        Parameters
        ----------
        weight_path : str
            Path to the trained RetinaNet weights (``.pth``).
        """
        print("--- Initializing Mouth Detector (Detectron2 mode) ---")

        # 1. Configure the model
        cfg = get_cfg()
        # Use a Detectron2 base config.
        cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_101_FPN_3x.yaml"))
        cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

        # 2. Load the weights
        cfg.MODEL.WEIGHTS = weight_path

        # 3. Adjust parameters specific to the trained model
        cfg.MODEL.RETINANET.NUM_CLASSES = 3  # Number of classes the model was trained on
        cfg.MODEL.RETINANET.SCORE_THRESH_TEST = 0.5 # Adjust the threshold as needed

        print(f"Selected inference device: {cfg.MODEL.DEVICE}")
        print(f"Loading weights from: {weight_path}")

        # 4. Create the "predictor" object
        try:
            self.predictor = DefaultPredictor(cfg)
            print("Detectron2 model loaded successfully!")
        except Exception as e:
            print(f"ERROR creating DefaultPredictor: {e}")
            self.predictor = None

    def predict(self, image_rgb: np.ndarray):
        """Detect the mouth in an RGB image and return the highest-scoring box.

        Parameters
        ----------
        image_rgb : np.ndarray
            Image in NumPy ``(H, W, C)`` layout with RGB channel order.

        Returns
        -------
        list of float or None
            Bounding box ``[xmin, ymin, xmax, ymax]`` for the highest-scoring
            detection, or ``None`` if no mouth is detected or the predictor is
            unavailable.
        """
        if self.predictor is None:
            print("ERROR: predictor was not loaded; cannot run prediction.")
            return None

        try:
            # The Detectron2 predictor expects a BGR image, so convert it back
            image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

            # Run prediction
            with torch.no_grad():
                outputs = self.predictor(image_bgr)

            # Extract the results
            instances = outputs["instances"].to("cpu")
            pred_boxes = instances.pred_boxes.tensor.numpy()
            scores = instances.scores.numpy()

            if len(scores) > 0:
                # Find the index of the highest score (best detection)
                max_score_idx = np.argmax(scores)
                best_box = pred_boxes[max_score_idx].tolist()
                best_score = scores[max_score_idx]

                return best_box
            else:
                return None

        except Exception as e:
            print(f"ERROR during Detectron2 inference: {e}")
            return None

class InferenciaSegmentacaoDeDentes:
    def __init__(self, weights_path, score_threshold=0.2):
        """Initialize the segmenter from the trained Mask R-CNN weights.

        Parameters
        ----------
        weights_path : str
            Path to the trained Detectron2 Mask R-CNN weights (``.pth``).
        score_threshold : float, optional
            Minimum confidence required to keep a detection.
        """
        print("[ToothSegmenter] Initializing via DefaultPredictor...")

        cfg = get_cfg()
        cfg.merge_from_file(model_zoo.get_config_file("Misc/scratch_mask_rcnn_R_50_FPN_9x_gn.yaml"))
        cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        cfg.MODEL.WEIGHTS = weights_path
        cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = score_threshold

        self.cfg = cfg
        self.predictor = DefaultPredictor(cfg)
        self.metadata = self._setup_metadata()

        print(f"[ToothSegmenter] Model '{weights_path}' loaded on {cfg.MODEL.DEVICE}.")

    def _setup_metadata(self):
        """Register placeholder metadata so Detectron2 visualizers do not crash."""
        try:
            return MetadataCatalog.get(self.cfg.DATASETS.TRAIN[0])
        except Exception:
            # Build generic metadata when the training dataset is not registered
            dataset_name = "dentes_metadata_temp"
            metadata = MetadataCatalog.get(dataset_name)
            metadata.thing_classes = ["dente"] # Defina a classe aqui
            return metadata

    def infer(self, cv2_image, visualize=False):
        """Run instance segmentation and return one simplified polygon per detection.

        Parameters
        ----------
        cv2_image : np.ndarray
            Image in OpenCV BGR layout.
        visualize : bool, optional
            If True, plot the predicted polygons over the input image.

        Returns
        -------
        list of np.ndarray
            One simplified contour per detection in cv2 polygon format.
        """
        print("Running inference...")
        outputs = self.predictor(cv2_image)
        predictions = outputs["instances"].to("cpu")

        # 'predictions.pred_masks' is already at full image resolution.
        # No need for 'paste_masks_in_image' or 'pred_boxes' here.
        full_masks_numpy = predictions.pred_masks.numpy()

        poligonos = self.extrair_poligonos_de_mascaras(full_masks_numpy)

        if visualize:
            print("Visualization enabled; drawing polygons...")
            self.visualize_polygons(cv2_image, poligonos)

        return poligonos

    def extrair_poligonos_de_mascaras(self, full_masks_numpy, epsilon_factor=0.008):
        """Convert full-resolution binary masks to simplified contour polygons.

        Parameters
        ----------
        full_masks_numpy : np.ndarray
            Stacked masks with shape ``(N, H, W)``.
        epsilon_factor : float, optional
            Douglas-Peucker simplification factor relative to the contour perimeter.
        """
        print(f"Processing {full_masks_numpy.shape[0]} full-resolution masks...")
        poligonos_simplificados = []
        for mascara_individual in full_masks_numpy:
            # Mask is already (HxW, bool); just convert to uint8.
            mascara_para_contorno = mascara_individual.astype(np.uint8) * 255

            contornos, _ = cv2.findContours(mascara_para_contorno, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not contornos:
                continue

            maior_contorno = max(contornos, key=cv2.contourArea)
            perimetro = cv2.arcLength(maior_contorno, True)
            epsilon = epsilon_factor * perimetro
            poligono_simplificado = cv2.approxPolyDP(maior_contorno, epsilon, True)
            poligonos_simplificados.append(poligono_simplificado)

        print(f"Extraction complete. {len(poligonos_simplificados)} polygons produced.")
        return poligonos_simplificados

    def visualize_polygons(self, image, polygons):
        """Plot a list of polygons over an image with Matplotlib.

        Parameters
        ----------
        image : np.ndarray
            Source image in OpenCV BGR layout.
        polygons : list of np.ndarray
            Contours in cv2 format.
        """
        # Copy the image so the original is not modified.
        image_with_polygons = image.copy()

        # Draw all polygons. The polygon list is already in the format
        # expected by cv2.drawContours.
        cv2.drawContours(image_with_polygons, contours=polygons, contourIdx=-1, color=(0, 255, 0), thickness=2)

        # Convert OpenCV BGR to RGB for correct Matplotlib display.
        image_rgb = cv2.cvtColor(image_with_polygons, cv2.COLOR_BGR2RGB)

        # Render with Matplotlib.
        plt.figure(figsize=(15, 10))  # Reasonable figure size
        plt.imshow(image_rgb)
        plt.title("Simplified tooth polygons")
        plt.axis('off') # Remove os eixos X e Y
        plt.show()

def process_and_detect_mouth(image_path, mouth_detector, visualize=False, pre_crop_margin_percentage=5):
    """
    Processes an image, applies a pre-cropping step with margin,
    detects the mouth in the pre-cropped image, and optionally visualizes the results.

    Args:
        image_path (str): Path to the input image.
        mouth_detector (MouthDetector): An instance of the MouthDetector class.
        visualize (bool): If True, plots the original and pre-cropped images side-by-side.
        pre_crop_margin_percentage (int or float): Percentage of the image width
                                                  to remove from the left and right sides
                                                  during the pre-cropping step.

    Returns:
        np.ndarray or None: The cropped image (based on mouth detection within the pre-cropped area)
                            if a mouth is detected, otherwise None.
    """

    if isinstance(image_path, str):
        original_image_bgr = cv2.imread(image_path)
        if original_image_bgr is None:
            print(f'Image could not be loaded: {image_path}')
            return None
    elif isinstance(image_path, np.ndarray):
        original_image_bgr = image_path
    else:
        raise TypeError("process_and_detect_mouth expects a file path (str) or a NumPy array.")

    h, w, _ = original_image_bgr.shape
    if h > 1080 or w > 1920:
        scale = min(1920 / w, 1080 / h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        image_to_process_bgr = cv2.resize(original_image_bgr.copy(), (new_w, new_h), interpolation=cv2.INTER_AREA)
        print(f"Image resized to: {new_w}x{new_h}")
    else:
        image_to_process_bgr = original_image_bgr.copy()


    h_proc, w_proc, _ = image_to_process_bgr.shape
    margin_pixels = int(w_proc * (pre_crop_margin_percentage / 100.0))

    pre_cropped_xmin = margin_pixels
    pre_cropped_xmax = w_proc - margin_pixels

    pre_cropped_xmin = max(0, pre_cropped_xmin)
    pre_cropped_xmax = min(w_proc, pre_cropped_xmax)

    image_pre_cropped_bgr = image_to_process_bgr[:, pre_cropped_xmin:pre_cropped_xmax]

    image_pre_cropped_rgb = cv2.cvtColor(image_pre_cropped_bgr, cv2.COLOR_BGR2RGB)

    box_in_pre_crop = mouth_detector.predict(image_pre_cropped_rgb)

    cropped_image = None
    if box_in_pre_crop:
        xmin_in_pre_crop, ymin_in_pre_crop, xmax_in_pre_crop, ymax_in_pre_crop = map(int, box_in_pre_crop)

        final_xmin = pre_cropped_xmin + xmin_in_pre_crop
        final_ymin = ymin_in_pre_crop
        final_xmax = pre_cropped_xmin + xmax_in_pre_crop
        final_ymax = ymax_in_pre_crop

        final_xmin = max(0, final_xmin)
        final_ymin = max(0, final_ymin)
        final_xmax = min(w_proc, final_xmax)
        final_ymax = min(h_proc, final_ymax)

        cropped_image = image_to_process_bgr[final_ymin:final_ymax, final_xmin:final_xmax]

    if visualize:
        fig, axes = plt.subplots(1, 2, figsize=(15, 8))

        axes[0].imshow(cv2.cvtColor(image_to_process_bgr, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original Image (Processed Resolution)")
        axes[0].axis('off')

        axes[1].imshow(cv2.cvtColor(cropped_image, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Post-Cropped Image ({pre_crop_margin_percentage}% Margin)")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

    return cropped_image


class BoneLossSegmentation:
    def __init__(self, model_crowns_path, model_bone_path):
        self.model_crowns = None
        self.model_bone = None
        print("[BoneLossSegmentation] Loading models...")
        try:
            self.model_crowns = YOLO(model_crowns_path)
            self.model_bone = YOLO(model_bone_path)
            print("[BoneLossSegmentation] Models loaded successfully!")
        except Exception as e:
            print(f"[BoneLossSegmentation] An error occurred while loading the models: {e}")
            return None, None


    def process_and_visualize_segmentation(
        self,
        image: np.ndarray,
        visualize: bool = True,
        conf_threshold: float = 0.25
    ):
        """
        Processes an image object (NumPy array) with two separate YOLO segmentation models
        and optionally visualizes the results.

        Args:
            image (np.ndarray): The input image as a NumPy array (e.g., from cv2.imread).
            model_crowns_path (str): The path to the .pt model for tooth crown segmentation.
            model_bone_path (str): The path to the .pt model for bone ridge segmentation.
            visualize (bool, optional): If True, displays the segmentation results. Defaults to True.
            conf_threshold (float, optional): Confidence threshold for predictions. Defaults to 0.25.

        Returns:
            tuple: A tuple containing the results from the crowns model and the bone model.
                  (results_crowns, results_bone)
        """
        # --- 1. Input Validation ---
        if not isinstance(image, np.ndarray):
            print("Error: The provided image must be a NumPy array.")
            return None, None

        print("[BoneLossSegmentation] Running inference on image")

        results_crowns = self.model_crowns.predict(source=image, imgsz=(736, 1280), conf=conf_threshold, show_conf=False, show_boxes=False)
        results_bone = self.model_bone.predict(source=image, imgsz=(736, 1280), conf=conf_threshold)
        print("[BoneLossSegmentation] Inference complete.")

        if visualize:
            print("--- Visualizando resultados... ---")
            plotted_crowns_bgr = results_crowns[0].plot(boxes=False, conf=False)
            plotted_bone_bgr = results_bone[0].plot(boxes=False, conf=False)

            plotted_crowns_rgb = cv2.cvtColor(plotted_crowns_bgr, cv2.COLOR_BGR2RGB)
            plotted_bone_rgb = cv2.cvtColor(plotted_bone_bgr, cv2.COLOR_BGR2RGB)

            fig, axes = plt.subplots(1, 2, figsize=(20, 10)) # 1 row, 2 columns

            axes[0].imshow(plotted_crowns_rgb)
            axes[0].set_title("Model: Coroas (Crowns)")
            axes[0].axis('off')

            axes[1].imshow(plotted_bone_rgb)
            axes[1].set_title("Model: Alveolar Ridge")
            axes[1].axis('off')

            plt.suptitle('Segmentation Results', fontsize=16)
            plt.tight_layout()
            plt.show()

        return results_crowns, results_bone

def parse_yolo_results(results):
    parsed = []
    boxes = results[0].boxes
    masks = results[0].masks.xy if results[0].masks else None

    for i, r in enumerate(boxes):
        box = r.xyxy[0].cpu().numpy().tolist()
        conf = r.conf.item()
        cls = int(r.cls.item())
        name = results[0].names[cls]

        segment = None
        if masks is not None and i < len(masks):
            seg = masks[i]
            segment = {
                'x': [float(p[0]) for p in seg],
                'y': [float(p[1]) for p in seg]
            }

        parsed.append({
            'name': name,
            'class': cls,
            'confidence': conf,
            'box': {'x1': box[0], 'y1': box[1], 'x2': box[2], 'y2': box[3]},
            'segments': segment
        })
    return parsed

def analisar_perda_ossea(dentes_poligonos, crown_results, bone_results,
                         smooth_distance=5.5, min_area_threshold=10):
    """Compute the area, geometry, and percentage of bone loss for each tooth.

    Discards degenerate geometries (zero area) that would otherwise raise
    Shapely topology errors. Pure computation: this function does not draw.
    """
    resultados_info = []
    perda_ossea_shapes = []

    try:
        # 1. Build crown polygons with two validation layers.
        coroas_polygons = []
        for c in crown_results:
            # Ensure enough points to form a polygon.
            if 'segments' in c and c['segments'] and len(c['segments']['x']) > 2:
                poly = Polygon(zip(c['segments']['x'], c['segments']['y']))

                # First validation layer: fix invalid topologies.
                if not poly.is_valid:
                    poly = poly.buffer(0)

                # Second validation layer: ensure non-zero area.
                if poly and not poly.is_empty and poly.area > 0:
                    coroas_polygons.append(poly)

        if not coroas_polygons:
             print("[Warning] No crown with valid geometry remained after filtering; skipping bone-loss analysis for this image.")
             return [], []

        coroa_union = unary_union(coroas_polygons)

        if not bone_results or 'segments' not in bone_results[0] or not bone_results[0]['segments']['x']:
            raise ValueError("Alveolar-ridge data missing or invalid.")

        rebordo_polygon = Polygon(zip(bone_results[0]['segments']['x'], bone_results[0]['segments']['y']))
        if not rebordo_polygon.is_valid:
            rebordo_polygon = rebordo_polygon.buffer(0)

        # 2. Iterate over each tooth.
        for i, dente_poly_np in enumerate(dentes_poligonos):
            info = {"indice_dente": i, "area_total_dente_px": 0, "area_perda_ossea_px": 0, "porcentagem_perda_ossea": 0.0}
            try:
                dente_poly = Polygon(dente_poly_np.reshape(-1, 2))
                if not dente_poly.is_valid:
                    dente_poly = dente_poly.buffer(0)

                if dente_poly.area == 0:
                    resultados_info.append(info)
                    perda_ossea_shapes.append(None)
                    continue

                info["area_total_dente_px"] = round(dente_poly.area, 2)
                dente_sem_coroa = dente_poly.difference(coroa_union)

                if dente_sem_coroa.is_empty:
                    resultados_info.append(info)
                    perda_ossea_shapes.append(None)
                    continue

                perda_ossea_poly_bruta = dente_sem_coroa.intersection(rebordo_polygon)
                perda_ossea_final = None
                if perda_ossea_poly_bruta and not perda_ossea_poly_bruta.is_empty:
                    poly_suavizado = perda_ossea_poly_bruta.buffer(-smooth_distance).buffer(smooth_distance)
                    if isinstance(poly_suavizado, MultiPolygon):
                        poligonos_filtrados = [p for p in poly_suavizado.geoms if p.area > min_area_threshold]
                        if poligonos_filtrados:
                            perda_ossea_final = unary_union(poligonos_filtrados)
                    elif poly_suavizado.area > min_area_threshold:
                        perda_ossea_final = poly_suavizado

                if perda_ossea_final:
                    info["area_perda_ossea_px"] = round(perda_ossea_final.area, 2)
                    if info["area_total_dente_px"] > 0:
                        info["porcentagem_perda_ossea"] = round((info["area_perda_ossea_px"] / info["area_total_dente_px"]) * 100, 2)

                perda_ossea_shapes.append(perda_ossea_final)
                resultados_info.append(info)

            except Exception as e_dente:
                print(f"[Warning] Failed to process tooth {i}: {e_dente}")
                resultados_info.append(info)
                perda_ossea_shapes.append(None)

        return resultados_info, perda_ossea_shapes

    except Exception as e_critico:
        print(f"\n[Critical Error] analisar_perda_ossea failed: {e_critico}")
        traceback.print_exc()
        return [], []


# --- LEGACY DEBUG FUNCTION (Matplotlib, no MultiPolygon handling) ---
# Kept for reference; this variant can raise AttributeError on MultiPolygon outputs.

def analisar_perda_ossea_debug_old(dentes_poligonos, crown_results, bone_results,
                                   visualize=False, image=None,
                                   smooth_distance=5.5, min_area_threshold=10):
    """Legacy debug variant that visualizes with Matplotlib but does not handle ``MultiPolygon`` outputs."""
    resultados_info = []
    perda_ossea_shapes = []
    visualized_image = None

    try:
        # 1. Build crown and ridge polygons.
        coroas_polygons = []
        for c in crown_results:
            if 'segments' in c and c['segments'] and len(c['segments']['x']) > 2:
                poly = Polygon(zip(c['segments']['x'], c['segments']['y']))
                if not poly.is_valid:
                    poly = poly.buffer(0)
                coroas_polygons.append(poly)

        coroa_union = unary_union(coroas_polygons)

        if not bone_results or 'segments' not in bone_results[0] or not bone_results[0]['segments']['x']:
            raise ValueError("Alveolar-ridge data missing or invalid.")
        rebordo_polygon = Polygon(zip(bone_results[0]['segments']['x'], bone_results[0]['segments']['y']))

        # 2. Iterate over each tooth.
        for i, dente_poly_np in enumerate(dentes_poligonos):
            info = {"indice_dente": i, "area_total_dente_px": 0, "area_perda_ossea_px": 0, "porcentagem_perda_ossea": 0.0}
            try:
                dente_poly = Polygon(dente_poly_np.reshape(-1, 2))
                if not dente_poly.is_valid or dente_poly.area == 0:
                    resultados_info.append(info); perda_ossea_shapes.append(None); continue
                info["area_total_dente_px"] = round(dente_poly.area, 2)
                dente_sem_coroa = dente_poly.difference(coroa_union)
                if dente_sem_coroa.is_empty:
                    resultados_info.append(info); perda_ossea_shapes.append(None); continue
                perda_ossea_poly_bruta = dente_sem_coroa.intersection(rebordo_polygon)
                perda_ossea_final = None
                if perda_ossea_poly_bruta and not perda_ossea_poly_bruta.is_empty:
                    poly_suavizado = perda_ossea_poly_bruta.buffer(-smooth_distance).buffer(smooth_distance)
                    if isinstance(poly_suavizado, MultiPolygon):
                        poligonos_filtrados = [p for p in poly_suavizado.geoms if p.area > min_area_threshold]
                        if poligonos_filtrados: perda_ossea_final = unary_union(poligonos_filtrados)
                    elif poly_suavizado.area > min_area_threshold:
                        perda_ossea_final = poly_suavizado
                if perda_ossea_final:
                    info["area_perda_ossea_px"] = round(perda_ossea_final.area, 2)
                    if info["area_total_dente_px"] > 0:
                        info["porcentagem_perda_ossea"] = round((info["area_perda_ossea_px"] / info["area_total_dente_px"]) * 100, 2)
                perda_ossea_shapes.append(perda_ossea_final)
                resultados_info.append(info)
            except Exception as e:
                print(f"[Error] Failed to process tooth {i}: {e}")
                resultados_info.append(info); perda_ossea_shapes.append(None)

        if visualize and image is not None:
            fig, ax = plt.subplots(figsize=(18, 12))
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            for dente_np in dentes_poligonos:
                coords = dente_np.reshape(-1, 2)
                ax.plot(np.append(coords[:, 0], coords[0, 0]), np.append(coords[:, 1], coords[0, 1]), color='blue', linewidth=1.5, alpha=0.7)
            for crown_poly in coroas_polygons:
                if crown_poly.is_valid: ax.plot(*crown_poly.exterior.xy, color='lime', linewidth=1.5, alpha=0.7)  # May fail on MultiPolygon
            if rebordo_polygon.is_valid: ax.plot(*rebordo_polygon.exterior.xy, color='orange', linewidth=2)
            patches = [MplPolygon(np.array(p.exterior.coords), closed=True) for shape in perda_ossea_shapes if shape and not shape.is_empty for p in (shape.geoms if isinstance(shape, MultiPolygon) else [shape])]
            ax.add_collection(PatchCollection(patches, facecolor='red', alpha=0.7, edgecolor='none'))
            for i, info in enumerate(resultados_info):
                porcentagem = info['porcentagem_perda_ossea']
                if porcentagem > 0:
                    dente_np = dentes_poligonos[i]
                    M = cv2.moments(dente_np)
                    if M["m00"] != 0:
                        cX, cY = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                        ax.text(cX, cY, f"{porcentagem:.1f}%", color='white', fontsize=8, ha='center', va='center', bbox=dict(facecolor='black', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2'))
            ax.set_title("Complete analysis with bone-loss percentage")
            ax.axis('off')
            fig.canvas.draw()
            buf = fig.canvas.buffer_rgba()
            img_array_rgba = np.asarray(buf)
            img_array_rgb = cv2.cvtColor(img_array_rgba, cv2.COLOR_RGBA2RGB)
            visualized_image = cv2.cvtColor(img_array_rgb, cv2.COLOR_RGB2BGR)
            plt.close(fig)
        return resultados_info, perda_ossea_shapes, visualized_image

    except Exception as e:
        print(f"\n[Critical Error] analisar_perda_ossea_debug_old failed: {e}")
        traceback.print_exc()
        return [], [], None


# --- DEBUG FUNCTION (Matplotlib, with MultiPolygon handling) ---

def analisar_perda_ossea_debug(dentes_poligonos, crown_results, bone_results,
                              visualize=False, image=None,
                              smooth_distance=5.5, min_area_threshold=10):
    """Debug variant of the bone-loss analysis with Matplotlib visualization.

    Handles both ``Polygon`` and ``MultiPolygon`` outputs from the geometric operations.
    """
    visualized_image = None
    resultados_info = []
    perda_ossea_shapes = []

    try:
        # Reuse the analysis function to obtain computed data.
        resultados_info, perda_ossea_shapes = analisar_perda_ossea(
            dentes_poligonos, crown_results, bone_results,
            smooth_distance, min_area_threshold
        )

        if visualize and image is not None:
            fig, ax = plt.subplots(figsize=(18, 12))
            ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

            # --- DRAWING LOGIC ---
            # Draw tooth contours.
            for dente_np in dentes_poligonos:
                coords = dente_np.reshape(-1, 2)
                ax.plot(np.append(coords[:, 0], coords[0, 0]), np.append(coords[:, 1], coords[0, 1]), color='blue', linewidth=1.5, alpha=0.7)

            # Draw crown polygons (handles MultiPolygon).
            coroas_polygons_para_desenho = [Polygon(zip(c['segments']['x'], c['segments']['y'])).buffer(0) for c in crown_results if 'segments' in c and c['segments'] and len(c['segments']['x']) > 2]
            for shape in coroas_polygons_para_desenho:
                if shape and not shape.is_empty:
                    polys_to_draw = shape.geoms if isinstance(shape, MultiPolygon) else [shape]
                    for poly in polys_to_draw:
                        if poly.is_valid:
                            ax.plot(*poly.exterior.xy, color='lime', linewidth=1.5, alpha=0.7)

            # Draw alveolar ridge.
            rebordo_polygon = Polygon(zip(bone_results[0]['segments']['x'], bone_results[0]['segments']['y']))
            if rebordo_polygon.is_valid:
                ax.plot(*rebordo_polygon.exterior.xy, color='orange', linewidth=2)
            else:  # Also handles the invalid-ridge case
                rebordo_fixed = rebordo_polygon.buffer(0)
                if rebordo_fixed.is_valid:
                    ax.plot(*rebordo_fixed.exterior.xy, color='orange', linewidth=2)

            # Fill the bone-loss area.
            patches = [MplPolygon(np.array(p.exterior.coords), closed=True) for shape in perda_ossea_shapes if shape and not shape.is_empty for p in (shape.geoms if isinstance(shape, MultiPolygon) else [shape])]
            ax.add_collection(PatchCollection(patches, facecolor='red', alpha=0.7, edgecolor='none'))

            # Add the percentage label.
            for i, info in enumerate(resultados_info):
                if info.get('porcentagem_perda_ossea', 0) > 0 and i < len(dentes_poligonos):
                    dente_np = dentes_poligonos[i]
                    M = cv2.moments(dente_np)
                    if M["m00"] != 0:
                        cX, cY = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                        ax.text(cX, cY, f"{info['porcentagem_perda_ossea']:.1f}%", color='white', fontsize=8, ha='center', va='center', bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.2'))
            # --- END DRAWING LOGIC ---

            ax.set_title("Complete analysis with bone-loss percentage")
            ax.axis('off')
            fig.canvas.draw()
            buf = fig.canvas.buffer_rgba()
            img_array_rgba = np.asarray(buf)
            img_array_rgb = cv2.cvtColor(img_array_rgba, cv2.COLOR_RGBA2RGB)
            visualized_image = cv2.cvtColor(img_array_rgb, cv2.COLOR_RGB2BGR)

            plt.close(fig)

        return resultados_info, perda_ossea_shapes, visualized_image

    except Exception as e:
        print(f"\n[Critical Error] analisar_perda_ossea_debug failed: {e}")
        traceback.print_exc()
        return [], [], None

def exportar_para_labelme(image_filename, image_array,
                          teeth_polygons, crown_results, bone_results, bone_loss_shapes,
                          output_json_path, simplification_factor=0.005):
    # ... (paste the full exportar_para_labelme function here) ...
    print(f"\nExporting LabelMe JSON to '{output_json_path}'...")
    img_h, img_w, _ = image_array.shape
    _, buffer = cv2.imencode('.jpg', image_array)
    image_data = base64.b64encode(buffer).decode('utf-8')
    shapes = []
    for poly in teeth_polygons:
        shapes.append({"label": "dente", "points": np.squeeze(poly).tolist(), "group_id": None, "shape_type": "polygon", "flags": {}})
    for result in crown_results:
        contour = np.array(list(zip(result['segments']['x'], result['segments']['y'])), dtype=np.int32)
        perimeter = cv2.arcLength(contour, True); epsilon = simplification_factor * perimeter
        simplified_contour = cv2.approxPolyDP(contour, epsilon, True)
        shapes.append({"label": "coroa", "points": np.squeeze(simplified_contour).tolist(), "group_id": None, "shape_type": "polygon", "flags": {}})
    if bone_results:
        result = bone_results[0]
        contour = np.array(list(zip(result['segments']['x'], result['segments']['y'])), dtype=np.int32)
        perimeter = cv2.arcLength(contour, True); epsilon = simplification_factor * perimeter
        simplified_contour = cv2.approxPolyDP(contour, epsilon, True)
        shapes.append({"label": "rebordo_osseo", "points": np.squeeze(simplified_contour).tolist(), "group_id": None, "shape_type": "polygon", "flags": {}})
    for shape in bone_loss_shapes:
        if shape is None or shape.is_empty: continue
        polygons_to_add = shape.geoms if hasattr(shape, 'geoms') else [shape]
        for poly in polygons_to_add:
            contour = np.array(poly.exterior.coords, dtype=np.int32).reshape((-1, 1, 2))
            perimeter = cv2.arcLength(contour, True); epsilon = simplification_factor * perimeter
            simplified_contour = cv2.approxPolyDP(contour, epsilon, True)
            shapes.append({"label": "perda_ossea", "points": np.squeeze(simplified_contour).tolist(), "group_id": None, "shape_type": "polygon", "flags": {}})
    labelme_data = {"version": "5.4.1", "flags": {}, "shapes": shapes, "imagePath": image_filename, "imageData": image_data, "imageHeight": img_h, "imageWidth": img_w}
    try:
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(labelme_data, f, ensure_ascii=False, indent=2)
        print(f"✅ Results (with simplified polygons) exported to {output_json_path}")
    except Exception as e:
        print(f"🚨 Failed to save JSON file: {e}")

def exportar_para_labelme_debug(image_path, image_array,
                          teeth_polygons, crown_results, bone_results, bone_loss_shapes,
                          output_json_path, simplification_factor=0.005):
    """Export all segmentation results to a LabelMe-compatible JSON file with simplified polygons.

    Parameters
    ----------
    simplification_factor : float
        Douglas-Peucker tolerance relative to each polygon's perimeter.
        Smaller values preserve more vertices.
    """
    print(f"\nExporting LabelMe JSON to '{output_json_path}'...")

    img_h, img_w, _ = image_array.shape
    _, buffer = cv2.imencode('.jpg', image_array)
    image_data = base64.b64encode(buffer).decode('utf-8')
    shapes = []

    for poly in teeth_polygons:
        shapes.append({
            "label": "tooth",
            "points": np.squeeze(poly).tolist(),
            "group_id": None, "shape_type": "polygon", "flags": {}
        })

    for result in crown_results:
        contour = np.array(list(zip(result['segments']['x'], result['segments']['y'])), dtype=np.int32)

        perimeter = cv2.arcLength(contour, True)
        epsilon = simplification_factor * perimeter
        simplified_contour = cv2.approxPolyDP(contour, epsilon, True)

        shapes.append({
            "label": "crown",
            "points": np.squeeze(simplified_contour).tolist(),
            "group_id": None, "shape_type": "polygon", "flags": {}
        })

    if bone_results:
        result = bone_results[0]
        contour = np.array(list(zip(result['segments']['x'], result['segments']['y'])), dtype=np.int32)

        perimeter = cv2.arcLength(contour, True)
        epsilon = simplification_factor * perimeter
        simplified_contour = cv2.approxPolyDP(contour, epsilon, True)

        shapes.append({
            "label": "alveolar_ridge",
            "points": np.squeeze(simplified_contour).tolist(),
            "group_id": None, "shape_type": "polygon", "flags": {}
        })

    for shape in bone_loss_shapes:
        if shape is None or shape.is_empty:
            continue

        polygons_to_add = shape.geoms if hasattr(shape, 'geoms') else [shape]
        for poly in polygons_to_add:
            contour = np.array(poly.exterior.coords, dtype=np.int32).reshape((-1, 1, 2))

            perimeter = cv2.arcLength(contour, True)
            epsilon = simplification_factor * perimeter
            simplified_contour = cv2.approxPolyDP(contour, epsilon, True)

            shapes.append({
                "label": "bone_loss",
                "points": np.squeeze(simplified_contour).tolist(),
                "group_id": None, "shape_type": "polygon", "flags": {}
            })

    labelme_data = {
        "version": "5.4.1",
        "flags": {},
        "shapes": shapes,
        "imagePath": os.path.basename(image_path),
        "imageData": image_data,
        "imageHeight": img_h,
        "imageWidth": img_w
    }

    try:
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(labelme_data, f, ensure_ascii=False, indent=2)
        print(f"✅ Results (with simplified polygons) exported to {output_json_path}")
    except Exception as e:
        print(f"🚨 Failed to save JSON file: {e}")

def gerar_imagem_visualizacao(image, dentes_poligonos, coroas_results, rebordo_results, perda_ossea_shapes, info_perda_ossea):
    image_copy = image.copy()
    coroas_polygons = [Polygon(zip(c['segments']['x'], c['segments']['y'])) for c in coroas_results if len(c['segments']['x']) > 2]
    rebordo_polygon = Polygon(zip(rebordo_results[0]['segments']['x'], rebordo_results[0]['segments']['y']))

    # Desenha contornos
    cv2.drawContours(image_copy, dentes_poligonos, -1, (255, 0, 0), 2) # Dentes em Azul
    for p in coroas_polygons: cv2.polylines(image_copy, [np.array(p.exterior.coords, dtype=np.int32)], True, (0, 255, 0), 2) # Coroas em Verde
    cv2.polylines(image_copy, [np.array(rebordo_polygon.exterior.coords, dtype=np.int32)], True, (0, 165, 255), 2)  # Ridge in orange

    # Fill the bone-loss area.
    overlay = image_copy.copy()
    for shape in perda_ossea_shapes:
        if shape and not shape.is_empty:
            polys = shape.geoms if hasattr(shape, 'geoms') else [shape]
            for p in polys:
                cv2.fillPoly(overlay, [np.array(p.exterior.coords, dtype=np.int32)], (0, 0, 255))  # Bone loss in red
    image_copy = cv2.addWeighted(overlay, 0.6, image_copy, 0.4, 0)

    # Add the percentage label.
    for i, info in enumerate(info_perda_ossea):
        if info['porcentagem_perda_ossea'] > 0:
            dente_np = dentes_poligonos[i]
            M = cv2.moments(dente_np); cX = int(M["m10"] / M["m00"]); cY = int(M["m01"] / M["m00"])
            texto = f"{info['porcentagem_perda_ossea']:.1f}%"
            cv2.putText(image_copy, texto, (cX - 15, cY + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

    return cv2.cvtColor(image_copy, cv2.COLOR_BGR2RGB)

def analisar_imagem_completa(imagem_entrada):
    if imagem_entrada is None:
        raise gr.Error("Please upload an image.")

    # Gradio passes the image in RGB; convert to BGR for OpenCV processing.
    imagem_bgr = cv2.cvtColor(imagem_entrada, cv2.COLOR_RGB2BGR)

    output_dir = 'outputs'
    os.makedirs(output_dir, exist_ok=True)

    try:
        # 1. Crop the mouth region (input is already a numpy array).
        cropped_mouth_image = process_and_detect_mouth(imagem_bgr, mouth_detector, pre_crop_margin_percentage=7)
        if cropped_mouth_image is None:
            cropped_mouth_image = imagem_bgr
            print("No mouth detected in the image.")

        # 2. Tooth segmentation.
        polygons = teeth_segmentator.infer(cropped_mouth_image, visualize=False)

        # 3. Crown and alveolar-ridge segmentation.
        crown_results, bone_results = boneloss_segmentator.process_and_visualize_segmentation(
            image=cropped_mouth_image, visualize=False, conf_threshold=0.25
        )
        if not crown_results or not bone_results:
            raise gr.Error("Crowns or alveolar ridge were not detected.")

        # 4. Parse YOLO results and compute bone loss.
        parsed_crowns = parse_yolo_results(crown_results)
        parsed_bone = parse_yolo_results(bone_results)
        info_perda_ossea, shapes_perda_ossea = analisar_perda_ossea(polygons, parsed_crowns, parsed_bone)

        # 5. Build the output visualization.
        imagem_resultado = gerar_imagem_visualizacao(
            cropped_mouth_image, polygons, parsed_crowns, parsed_bone, shapes_perda_ossea, info_perda_ossea
        )

        # 6. Build and save the LabelMe JSON for download.
        output_json_filename = "resultado_analise.json"
        output_json_path = os.path.join(output_dir, output_json_filename)
        exportar_para_labelme(
            image_filename="imagem_cortada.png",
            image_array=cropped_mouth_image,
            teeth_polygons=polygons,
            crown_results=parsed_crowns,
            bone_results=parsed_bone,
            bone_loss_shapes=shapes_perda_ossea,
            output_json_path=output_json_path
        )

        return imagem_resultado, output_json_path

    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"An error occurred during analysis: {e}")

def gerar_imagem_visualizacao_cv2(image, dentes_poligonos, coroas_results, rebordo_results, perda_ossea_shapes, info_perda_ossea):
    """Render an OpenCV-only visualization that handles both ``Polygon`` and ``MultiPolygon`` outputs across all geometry layers."""
    image_copy = image.copy()
    alpha = 0.6

    # --- DRAWING HELPER ---
    def draw_shape(img, shape, color, thickness, is_closed=True, fill_color=None):
        if shape is None or shape.is_empty:
            return

        # Universal logic for Polygon and MultiPolygon.
        polys_to_draw = shape.geoms if isinstance(shape, MultiPolygon) else [shape]

        for poly in polys_to_draw:
            points = np.array(poly.exterior.coords, dtype=np.int32)
            if fill_color:
                cv2.fillPoly(img, [points], fill_color)
            cv2.polylines(img, [points], is_closed, color, thickness)

    try:
        # Desenha contornos dos dentes
        cv2.drawContours(image_copy, dentes_poligonos, -1, (255, 0, 0), 2)  # Dentes em Azul

        # Desenha coroas
        coroas_shapes = [Polygon(zip(c['segments']['x'], c['segments']['y'])).buffer(0) for c in coroas_results if len(c['segments']['x']) > 2]
        for shape in coroas_shapes:
            draw_shape(image_copy, shape, (0, 255, 0), 2) # Coroas em Verde

        # Draw the alveolar ridge.
        rebordo_shape = Polygon(zip(rebordo_results[0]['segments']['x'], rebordo_results[0]['segments']['y'])).buffer(0)
        draw_shape(image_copy, rebordo_shape, (0, 165, 255), 2)  # Ridge in orange

        # Fill the bone-loss area with transparency.
        overlay = image_copy.copy()
        for shape in perda_ossea_shapes:
            if shape:
                 draw_shape(overlay, shape, color=None, thickness=None, fill_color=(0, 0, 255))  # Fill red
        cv2.addWeighted(overlay, alpha, image_copy, 1 - alpha, 0, image_copy)

        # Add the percentage label.
        for i, info in enumerate(info_perda_ossea):
            porcentagem = info.get('porcentagem_perda_ossea', 0)
            if porcentagem > 0 and i < len(dentes_poligonos):
                dente_np = dentes_poligonos[i]
                M = cv2.moments(dente_np)
                if M["m00"] != 0:
                    cX, cY = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                    texto = f"{porcentagem:.1f}%"
                    cv2.putText(image_copy, texto, (cX - 15, cY + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 4, cv2.LINE_AA)
                    cv2.putText(image_copy, texto, (cX - 15, cY + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        return image_copy

    except Exception as e:
        print(f"[OpenCV Visualization Error] Failed to draw geometries: {e}")
        return image  # Return the original image if drawing fails

def gerar_imagem_visualizacao_cv2_old(image, dentes_poligonos, coroas_results, rebordo_results, perda_ossea_shapes, info_perda_ossea):
    """Legacy OpenCV-only visualizer (no Matplotlib chrome).

    Produces a clean image without white borders or chart legends.
    """
    # Copy so we do not draw on the original image.
    image_copy = image.copy()
    alpha = 0.6  # Transparency for the bone-loss overlay

    # Polygon preparation.
    try:
        coroas_polygons = [np.array(list(zip(c['segments']['x'], c['segments']['y'])), dtype=np.int32) for c in coroas_results if len(c['segments']['x']) > 2]
        rebordo_polygon = np.array(list(zip(rebordo_results[0]['segments']['x'], rebordo_results[0]['segments']['y'])), dtype=np.int32)
    except (IndexError, KeyError) as e:
        print(f"[Visualization Warning] Could not extract crown or ridge polygons: {e}")
        return image_copy  # Return the un-annotated image when inputs are missing

    # Desenha contornos
    cv2.drawContours(image_copy, dentes_poligonos, -1, (255, 0, 0), 2)  # Dentes em Azul (BGR)
    cv2.polylines(image_copy, coroas_polygons, True, (0, 255, 0), 2)     # Coroas em Verde
    cv2.polylines(image_copy, [rebordo_polygon], True, (0, 165, 255), 2)  # Ridge in orange

    # Fill the bone-loss area with transparency.
    overlay = image_copy.copy()
    for shape in perda_ossea_shapes:
        if shape and not shape.is_empty:
            polys_para_desenhar = shape.geoms if hasattr(shape, 'geoms') else [shape]
            for p in polys_para_desenhar:
                pontos = np.array(p.exterior.coords, dtype=np.int32)
                cv2.fillPoly(overlay, [pontos], (0, 0, 255))  # Bone loss in red

    # Blend the overlay onto the original image.
    cv2.addWeighted(overlay, alpha, image_copy, 1 - alpha, 0, image_copy)

    # Add the percentage label.
    for i, info in enumerate(info_perda_ossea):
        porcentagem = info.get('porcentagem_perda_ossea', 0)
        if porcentagem > 0:
            dente_np = dentes_poligonos[i]
            M = cv2.moments(dente_np)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
                texto = f"{porcentagem:.1f}%"

                # Draw the text with a black outline for legibility.
                cv2.putText(image_copy, texto, (cX - 15, cY + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 4, cv2.LINE_AA)
                cv2.putText(image_copy, texto, (cX - 15, cY + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

    return image_copy

mouth_detector = MouthDetector('/content/Models/MouthDetection/inredd_mouth_detector.pth')

#teeth_segmentator_initializer = Segmentacao_De_Dentes()
teeth_segmentator = InferenciaSegmentacaoDeDentes('/content/Models/TeethSegmentation/inredd_tooth_segmenter.pth')

boneloss_segmentator = BoneLossSegmentation(
    model_crowns_path='/content/Models/BoneLossSegmentation/inredd_crown_segmenter.pt',
    model_bone_path='/content/Models/BoneLossSegmentation/inredd_alveolar_ridge_segmenter.pt'
  )

# 3. Interactive Analysis Tool

Run the cell below to launch the Gradio interface. Upload a panoramic radiograph, click **Run analysis**, and the right-hand panel displays the cropped mouth region with overlaid annotations (teeth, crowns, alveolar ridge, and bone-loss area). A LabelMe-compatible JSON file is also produced for download.

> Gradio prints a public URL when launched (`Running on public URL: ...`); this URL stays valid as long as the cell is running. Stopping the cell or closing the runtime invalidates the link.


In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Dental Radiograph Analyzer") as demo:
    gr.Markdown("# Bone-Loss Analyzer for Panoramic Radiographs")
    gr.Markdown("Upload a panoramic radiograph to detect the mouth, teeth, crowns, alveolar ridge, and to compute periodontal bone loss.")

    with gr.Row():
        image_input = gr.Image(type="numpy", label="Upload Radiograph")
        with gr.Column():
            image_output = gr.Image(label="Analysis Result", interactive=False, format="png")
            json_output = gr.File(label="Download LabelMe annotation (.json)")

    run_button = gr.Button("▶Run analysis", variant="primary")

    run_button.click(
        fn=analisar_imagem_completa,
        inputs=image_input,
        outputs=[image_output, json_output]
    )

    gr.Markdown("---")
    gr.Markdown("Mouth-detection and tooth-segmentation models developed by an InReDD researcher.")
    gr.Markdown("Crown- and alveolar-ridge-segmentation models developed by an InReDD researcher.")
    gr.Markdown("Results are for research purposes only and do not replace professional clinical diagnosis.")

demo.launch(debug=True)

# 4. Debug Mode (optional)

> **For methodological inspection only.** This section is independent of the Gradio interface and is not required for normal use.

Each pipeline stage can be run in isolation by setting `visualize=True` on the corresponding call (e.g. `teeth_segmentator.infer(..., visualize=True)`) and leaving the others as `visualize=False`. The cell below iterates over a directory of images (`input_dir`) and runs the full pipeline, writing cropped images, visualizations, and LabelMe JSON files to `output_dir`.


In [ ]:
if __name__ == '__main__':
    try:
        input_dir = '/content/Images_to_validate_PAN_boneloss'
        output_dir = '/content/outputs'
        os.makedirs(output_dir, exist_ok=True)

        image_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
        image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(image_extensions)]
        total_images = len(image_files)
        if total_images == 0:
            print(f"No images found in directory: {input_dir}")
        else:
            print(f"\nStarting processing of {total_images} images...")
        for i, filename in enumerate(sorted(image_files)):
            print(f"\n{'='*20} [ Processing image {i+1}/{total_images}: {filename} ] {'='*20}")
            image_path = os.path.join(input_dir, filename)


            base_name = os.path.splitext(os.path.basename(image_path))[0]

            cropped_mouth_image = process_and_detect_mouth(image_path, mouth_detector, visualize=False, pre_crop_margin_percentage=7)

            h, w, _ = cropped_mouth_image.shape

            if cropped_mouth_image is not None:
                print("Mouth detected; image cropped.")
                cropped_image_filename = f"{base_name}_output.png"
                cropped_image_path = os.path.join(output_dir, cropped_image_filename)
                cv2.imwrite(cropped_image_path, cropped_mouth_image)
                print(f"Cropped mouth image saved to: {cropped_image_path}")
            else:
                print("No mouth detected.")

            polygons = teeth_segmentator.infer(cropped_mouth_image, visualize=False)
            # print("Detectron tooth polygons output", polygons)

            crown_results, bone_results = boneloss_segmentator.process_and_visualize_segmentation(
                image=cropped_mouth_image,
                visualize=False,
                conf_threshold=0.25
            )

            if crown_results and bone_results:
                print("\nCrowns Model Results Summary:")
                print(crown_results[0].summary())
                print("\nAlveolar Ridge Model Results Summary:")
                print(bone_results[0].summary())

                parsed_crowns = parse_yolo_results(crown_results)
                parsed_bone = parse_yolo_results(bone_results)

                info_perda_ossea, shapes_perda_ossea, image_output  = analisar_perda_ossea_debug(
                    polygons,
                    parsed_crowns,
                    parsed_bone,
                    visualize=True,
                    image=cropped_mouth_image
                )
                image_output = gerar_imagem_visualizacao_cv2(
                    cropped_mouth_image,
                    polygons,
                    parsed_crowns,
                    parsed_bone,
                    shapes_perda_ossea,
                    info_perda_ossea
                )

                if image_output is not None:
                    output_image_filename = f"{base_name}_analysis_visualization.png"
                    output_image_path = os.path.join(output_dir, output_image_filename)
                    cv2.imwrite(output_image_path, image_output)
                    print(f"Analysis visualization saved to: {output_image_path}")
                else:
                    print("No visualization image was produced.")
            else:
                print("\nFunction execution failed. Check file paths and model integrity.")
        print(f"\n{'='*20} All images processed. {'='*20}")
    except Exception as e:
        print("\nAn error occurred during execution:")
        print(f"Error type: {type(e).__name__}")
        print(f"Message: {e}")

        # Automated diagnostics
        if isinstance(e, FileNotFoundError):
            print("Check whether the image or model path is correct.")
        elif isinstance(e, IndexError):
            print("Results may be empty; check whether the model actually produced predictions.")
        elif isinstance(e, AttributeError):
            print("An expected variable may be None; check that all models loaded correctly.")
        elif isinstance(e, ValueError):
            print("An unexpected value was passed; check the model input/output formats.")
        elif "placeholder" in str(e).lower():
            print("This is expected when using placeholder `.pt` files.")
        else:
            print("ℹUnexpected error; review the inputs and the model/file formats.")

        print("\nSuggestions:")
        print("- Verify that the YOLO models were trained and loaded correctly.")
        print("- Verify the paths to images and `.pt` files.")
        print("- Use `print` or a debugger to inspect intermediate variables such as `crown_results`, `bone_results`, `polygons`.")

        print("\nFull traceback:")
        traceback.print_exc()